# Pulse Run VR: Section 4 analysis

This notebook runs the Meta Quest 3 primary analysis and the Meta Quest 2 + Meta Quest 3 sensitivity analysis. Statistical logic is stored in the `pulse_run_analysis` module so each calculation has one audited implementation.

Outputs remain marked **PROVISIONAL** until the headset model and session type have been verified for every otherwise eligible run and `strict_manual_review` is set to `true` in `analysis_config.toml`.

In [ ]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass
    class Image:
        def __init__(self, filename, width=None):
            self.filename = filename
        def __repr__(self):
            return self.filename
    def display(value):
        print(value)

from pulse_run_analysis import refresh_review_templates, run_complete_analysis

CONFIG_PATH = Path('analysis_config.toml')
OUTPUT_DIR = Path('outputs')

## 1. Refresh the review sheets

This appends new source files and run identifiers without replacing completed classifications. Fill the CSV files in `review/` before switching to final mode.

In [ ]:
manifest_path, review_path = refresh_review_templates(CONFIG_PATH)
display(pd.read_csv(manifest_path, keep_default_na=False))
display(pd.read_csv(review_path, keep_default_na=False).head())

## 2. Run validation, screening, derivation, statistics, tables, and figures

In [ ]:
complete = run_complete_analysis(CONFIG_PATH)
result = complete.primary
sensitivity = complete.sensitivity
{'primary': result.status, 'sensitivity': sensitivity.status}

## 3. Record flow and validation

In [ ]:
display(result.record_flow)
display(result.validation_report)

## 4. Tables 4-7

In [ ]:
for number, table in [(4, result.table_4), (5, result.table_5), (6, result.table_6), (7, result.table_7)]:
    display(Markdown(f'### Table {number}'))
    display(table)

## 5. Primary and sensitivity results

In [ ]:
display(Markdown('### Meta Quest 3 primary analysis'))
display(result.omnibus_tests)
display(result.pairwise_tests)
display(Markdown('### Meta Quest 2 + Meta Quest 3 sensitivity analysis'))
display(sensitivity.headset_distribution)
display(sensitivity.comparison_with_primary)
display(sensitivity.omnibus_tests)
display(sensitivity.pairwise_tests)

## 6. Manuscript figures

In [ ]:
for figure_name in [
    'figure3_pattern_composition.png',
    'figure4_performance_distributions.png',
    'figure5_post_run_ratings.png',
]:
    display(Image(filename=str(OUTPUT_DIR / 'figures' / figure_name), width=1000))

## 7. Diagnostic review

These files support assumption and influence checks. Flags are for inspection only and never trigger automatic record removal.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'diagnostics' / 'diagnostic_qq_residuals.png'), width=1000))
outlier_flags = pd.read_csv(OUTPUT_DIR / 'diagnostics' / 'outlier_flags.csv')
display(outlier_flags.loc[outlier_flags['outside_1_5_iqr'] | outlier_flags['absolute_residual_at_least_3']])
leave_one_out = pd.read_csv(OUTPUT_DIR / 'diagnostics' / 'leave_one_out_influence.csv')
display(leave_one_out.reindex(leave_one_out['change_in_p'].abs().sort_values(ascending=False).index).head(20))

## 8. Reporting text and software versions

In [ ]:
display(Markdown((OUTPUT_DIR / 'tables' / 'statistical_sentences.md').read_text(encoding='utf-8')))
display(Markdown((sensitivity.output_dir / 'tables' / 'statistical_sentences_primary.md').read_text(encoding='utf-8')))
display(pd.read_csv(OUTPUT_DIR / 'software_versions.csv'))